# UD5.04. TechStore: cuando la red no gana

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloque 9 de los apuntes · Criterios **2.c** y **2.e**

---

Este cuaderno tiene un resultado incómodo y es el más importante de la unidad:

> **Sobre el caso de abandono de TechStore, una red neuronal no gana al punto de
> referencia de una línea de la UD3.**

No es un fallo del guion, ni una semilla desafortunada, ni un error tuyo. Es el resultado,
está medido, y es reproducible. Y es la respuesta al criterio 2.c —*definir el modelo que
se quiere implementar según el problema planteado*—, porque esa respuesta no se da con una
preferencia: se da comparando con el punto de referencia.

Aquí se mide con cuidado, se explica por qué pasa, y se comprueba que **no** pasa en el
proyecto de la unidad, donde se clasifican imágenes y la misma comparación da lo
contrario.

**Este cuaderno usa `scikit-learn`.** Es la primera vez en el módulo, y es a propósito: en
la UD4 estaba prohibido porque había que construir a mano las curvas para entender qué
son. Ahora que se han construido, se llaman por su nombre.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score, precision_score,
                             recall_score, confusion_matrix)
from sklearn.calibration import calibration_curve

SEMILLA = 20262027
keras.utils.set_random_seed(SEMILLA)
print("keras", keras.__version__)

Cada función de `sklearn.metrics` que se usa aquí tiene detrás un gráfico que construiste
a mano en el cuaderno `UD4_05`. Conviene tener presente la equivalencia:

| Lo que se llama ahora | Lo que programaste en la UD4 |
|---|---|
| `roc_curve` | el `argsort` por puntuación y los dos `cumsum` |
| `roc_auc_score` | el área por la regla del trapecio |
| `confusion_matrix` | los cuatro recuentos con máscaras booleanas |
| `calibration_curve` | el diagrama de fiabilidad por tramos |

---

## 1. Los datos, tal como los dejó la UD3

In [ ]:
CARACTERISTICAS = ["recencia_dias", "frecuencia", "monetario", "ticket_medio",
                   "antiguedad_dias", "categorias_distintas", "proporcion_movil",
                   "dias_entre_pedidos", "cat_informatica", "cat_telefonia"]

RUTA_DATOS = os.path.join("datos", "clientes_abandono.csv")
if not os.path.exists(RUTA_DATOS):
    RUTA_DATOS = ("https://raw.githubusercontent.com/RafaSalaEsteve/"
                  "IABD-PIA-notebooks/main/UD5_tensores_redes_neuronales/"
                  "datos/clientes_abandono.csv")

tabla = pd.read_csv(RUTA_DATOS)
entrena = tabla[tabla["particion"] == "entrena"]
prueba = tabla[tabla["particion"] == "prueba"]

X_ent = entrena[CARACTERISTICAS].to_numpy(dtype="float32")
X_pru = prueba[CARACTERISTICAS].to_numpy(dtype="float32")
y_ent = entrena["abandona"].to_numpy(dtype="float32")
y_pru = prueba["abandona"].to_numpy(dtype="float32")

# La media y la desviacion, SOLO del entrenamiento. La regla de la UD3.
media, desv = X_ent.mean(axis=0), X_ent.std(axis=0)
desv[desv == 0] = 1.0
X_ent, X_pru = (X_ent - media) / desv, (X_pru - media) / desv

print(f"entrena {X_ent.shape}  abandona el {y_ent.mean() * 100:.1f} %")
print(f"prueba  {X_pru.shape}  abandona el {y_pru.mean() * 100:.1f} %")
print()
print("Poblacion entera: abandona el", f"{tabla['abandona'].mean() * 100:.1f} %")
print()
print("La particion es TEMPORAL: entrenan los clientes mas antiguos y se prueba")
print("con los mas recientes. Por eso las dos tasas no coinciden, y esa")
print("diferencia es un desplazamiento real, no un defecto del conjunto.")

### El aviso sobre el AUC de la UD4

La UD4 cita un AUC de **0,8484** para el punto de referencia de la recencia y **0,8562**
para su modelo A. Esas dos cifras están medidas **sobre los 400 clientes**. En este
cuaderno todo se mide sobre los **100 de la partición de prueba**, que no son los mismos
clientes ni tienen la misma tasa de abandono.

> **Comparar cifras medidas sobre poblaciones distintas es el error más fácil de cometer
> en esta unidad.** Vamos a imprimir las dos, para que la diferencia se vea y no se
> confunda con un resultado.

In [ ]:
recencia_todos = tabla["recencia_dias"].to_numpy(dtype="float64")
recencia_prueba = prueba["recencia_dias"].to_numpy(dtype="float64")

print(f"AUC de la recencia sobre los 400 clientes:  "
      f"{roc_auc_score(tabla['abandona'], recencia_todos):.4f}   <- la cifra de la UD4")
print(f"AUC de la recencia sobre los 100 de prueba: "
      f"{roc_auc_score(y_pru, recencia_prueba):.4f}   <- la de este cuaderno")

---

## 2. El punto de referencia, otra vez

*Se va quien lleva más de N días sin comprar.* Una columna y un umbral. Es lo que hay que
superar.

In [ ]:
print(f"{'umbral':>8} {'precision':>10} {'exhaust.':>10} {'F1':>8}")
print("-" * 40)
mejor = (0.0, None)
for umbral in (30, 60, 90, 120, 150, 180, 240):
    pred = (recencia_prueba > umbral).astype(int)
    p_ = precision_score(y_pru, pred, zero_division=0)
    e_ = recall_score(y_pru, pred, zero_division=0)
    f_ = f1_score(y_pru, pred, zero_division=0)
    print(f"{umbral:>8} {p_:>10.3f} {e_:>10.3f} {f_:>8.3f}")
    if f_ > mejor[0]:
        mejor = (f_, umbral)

UMBRAL_REF, F1_REF = mejor[1], mejor[0]
AUC_REF = roc_auc_score(y_pru, recencia_prueba)
print()
print(f"Punto de referencia: recencia > {UMBRAL_REF} dias   F1 {F1_REF:.3f}   AUC {AUC_REF:.3f}")

---

## 3. Cuatro modelos, la misma partición

Del más pequeño al más grande, que es el orden del bloque 8.2. Cada uno se mide sobre el
entrenamiento **y** sobre la prueba, porque la distancia entre las dos cifras es lo que
interesa.

In [ ]:
def construye(capas, dropout=0.0, l2=0.0):
    reg = keras.regularizers.l2(l2) if l2 else None
    m = keras.Sequential([keras.layers.Input(shape=(X_ent.shape[1],))])
    for unidades in capas:
        m.add(keras.layers.Dense(unidades, activation="relu", kernel_regularizer=reg))
        if dropout:
            m.add(keras.layers.Dropout(dropout))
    m.add(keras.layers.Dense(1, activation="sigmoid"))
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(name="auc")])
    return m


def mejor_umbral(y, p):
    # El umbral que maximiza F1, buscado sobre las propias puntuaciones.
    candidatos = np.unique(p)
    f1s = [f1_score(y, (p >= u).astype(int), zero_division=0) for u in candidatos]
    i = int(np.argmax(f1s))
    return float(candidatos[i]), float(f1s[i])


def entrena(nombre, capas, dropout=0.0, l2=0.0, parada=False, epocas=200):
    keras.utils.set_random_seed(SEMILLA)
    m = construye(capas, dropout, l2)
    llamadas = []
    if parada:
        llamadas.append(keras.callbacks.EarlyStopping(
            monitor="val_auc", mode="max", patience=30, restore_best_weights=True))
    t0 = time.perf_counter()
    h = m.fit(X_ent, y_ent, epochs=epocas, batch_size=32, validation_split=0.2,
              verbose=0, callbacks=llamadas)
    segundos = time.perf_counter() - t0

    p_pru = m.predict(X_pru, verbose=0).ravel()
    p_ent = m.predict(X_ent, verbose=0).ravel()
    umbral, f1 = mejor_umbral(y_pru, p_pru)
    return m, h, p_pru, {
        "modelo": nombre,
        "parametros": m.count_params(),
        "epocas": len(h.history["loss"]),
        "AUC entrena": roc_auc_score(y_ent, p_ent),
        "AUC prueba": roc_auc_score(y_pru, p_pru),
        "hueco": roc_auc_score(y_ent, p_ent) - roc_auc_score(y_pru, p_pru),
        "F1 mejor umbral": f1,
        "segundos": segundos,
    }

In [ ]:
resultados, modelos, historias, puntuaciones = [], {}, {}, {}

configuraciones = [
    ("logistica (sin capa oculta)", [],       0.0, 0.0,  False),
    ("densa 16-8, 200 epocas",      [16, 8],  0.0, 0.0,  False),
    ("densa 16-8 regularizada",     [16, 8],  0.3, 1e-4, True),
    ("densa 4, parada temprana",    [4],      0.0, 0.0,  True),
]

for nombre, capas, dr, l2, parada in configuraciones:
    m, h, p, fila = entrena(nombre, capas, dr, l2, parada)
    modelos[nombre], historias[nombre], puntuaciones[nombre] = m, h, p
    resultados.append(fila)
    aviso = ""
    if parada and fila["epocas"] == 200:
        aviso = "  <- la parada NO ha saltado: llego al limite de epocas"
    print(f"{nombre:30} {fila['segundos']:5.1f} s   {fila['epocas']:>3} epocas{aviso}")

In [ ]:
referencia = {"modelo": f"referencia UD3 (recencia > {UMBRAL_REF} d)",
              "parametros": 1, "epocas": 0,
              "AUC entrena": np.nan, "AUC prueba": AUC_REF, "hueco": np.nan,
              "F1 mejor umbral": F1_REF, "segundos": 0.0}

resumen = pd.DataFrame([referencia] + resultados)
pd.set_option("display.width", 160)
print(resumen.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

### Las tres lecturas

**1. Ninguna red gana al punto de referencia.** La logística se queda a milésimas; todo lo
que tiene capa oculta pierde entre cinco y ocho centésimas de AUC.

**2. El hueco es la medida del sobreajuste.** La logística tiene 0,018; la densa sin
precauciones, 0,178. No hace falta ninguna teoría para verlo: son dos números.

**3. La regularización funciona, y no basta.** Estrecha el hueco a la mitad y recupera casi
tres centésimas de AUC de prueba. Sigue por debajo del punto de referencia.

Y hay un detalle en la columna `epocas` que conviene no pasar por alto: en la red de 4
unidades **la parada temprana no llega a saltar**, porque con `patience=30` sobre
`val_auc` el AUC de validación sigue dando pequeños repuntes hasta el final. Una parada
temprana configurada con demasiada paciencia es una parada temprana que no existe, y eso
se detecta mirando si el número de épocas coincide con el máximo.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel 1: las curvas ROC.
ejes[0].plot([0, 1], [0, 1], "--", color="0.7", lw=1, label="azar")
fpr, tpr, _ = roc_curve(y_pru, recencia_prueba)
ejes[0].plot(fpr, tpr, lw=2.5, color="black",
             label=f"referencia UD3 (AUC {AUC_REF:.3f})")
for nombre, p in puntuaciones.items():
    fpr, tpr, _ = roc_curve(y_pru, p)
    ejes[0].plot(fpr, tpr, lw=1.5, alpha=0.85,
                 label=f"{nombre} ({roc_auc_score(y_pru, p):.3f})")
ejes[0].set_title("Ninguna curva pasa por encima de la negra")
ejes[0].set_xlabel("proporcion de falsas alarmas")
ejes[0].set_ylabel("proporcion de aciertos")
ejes[0].legend(fontsize=7, loc="lower right")

# Panel 2: el hueco de generalizacion.
orden = [f["modelo"] for f in resultados]
x = np.arange(len(orden))
ejes[1].bar(x - 0.2, [f["AUC entrena"] for f in resultados], 0.4,
            label="AUC entrenamiento")
ejes[1].bar(x + 0.2, [f["AUC prueba"] for f in resultados], 0.4,
            label="AUC prueba")
ejes[1].axhline(AUC_REF, color="black", ls="--", lw=1.5,
                label="punto de referencia")
ejes[1].set_xticks(x)
ejes[1].set_xticklabels([n.replace(", ", ",\n").replace(" (", "\n(") for n in orden],
                        fontsize=7)
ejes[1].set_ylim(0.5, 1.0)
ejes[1].set_ylabel("AUC")
ejes[1].set_title("La distancia entre las dos barras es el sobreajuste")
ejes[1].legend(fontsize=7)

fig.tight_layout()
plt.show()

---

## 4. Por qué pasa esto

Tres razones, y las tres se comprueban.

### 4.1. Más parámetros que datos

In [ ]:
n_entrenamiento = int(len(X_ent) * 0.8)      # el 20 % se va a validacion
print(f"Filas de entrenamiento reales: {n_entrenamiento}")
print()
print(f"{'modelo':30} {'parametros':>11} {'params/fila':>12}")
print("-" * 56)
for fila in resultados:
    print(f"{fila['modelo']:30} {fila['parametros']:>11,} "
          f"{fila['parametros'] / n_entrenamiento:>12.2f}")
print()
print("Con mas cosas que ajustar que datos con los que ajustarlas, memorizar")
print("es mas facil que generalizar. Y memorizar es exactamente lo que mide")
print("el hueco entre las dos columnas de AUC.")

### 4.2. Las características ya están construidas a mano

Recencia, frecuencia, monetario, ticket medio, antigüedad, días entre pedidos: eso es el
trabajo de la UD3, y es **exactamente** el trabajo que una red profunda hace por ti cuando
no está hecho.

Aquí ya está hecho. No queda representación que aprender; solo queda combinar diez números,
y para combinar diez números linealmente no hace falta una capa oculta. Se puede comprobar
mirando los pesos de la logística: son interpretables, y dicen lo que ya sabíamos.

In [ ]:
logistica = modelos["logistica (sin capa oculta)"]
pesos = logistica.get_weights()[0].ravel()

importancia = (pd.DataFrame({"caracteristica": CARACTERISTICAS, "peso": pesos})
               .assign(magnitud=lambda d: d["peso"].abs())
               .sort_values("magnitud", ascending=False))

print(importancia[["caracteristica", "peso"]].to_string(index=False,
                                                        float_format=lambda v: f"{v:+.3f}"))
print()
print("Las caracteristicas estan normalizadas, asi que los pesos son comparables.")
print("La recencia domina, con signo positivo: mas dias sin comprar, mas probabilidad")
print("de abandono. Es literalmente el punto de referencia de la UD3, aprendido.")

### 4.3. Una tercera explicación que la medición descarta

Hay una tercera sospecha razonable, y conviene seguirla hasta el final porque enseña más
al descartarse que al confirmarse.

La partición es temporal: entrenan los clientes más antiguos y se prueba con los más
recientes, que abandonan al 18 % frente al 26 % del entrenamiento. Esa diferencia es un
**desplazamiento de distribución** real, y es sensato sospechar que castiga más a un
modelo flexible que a una regla de una línea.

Se comprueba haciendo **la trampa a propósito**: partir al azar en lugar de por tiempo. Si
la sospecha fuera cierta, el AUC subiría de forma clara.

In [ ]:
from sklearn.model_selection import train_test_split

X_todo = tabla[CARACTERISTICAS].to_numpy(dtype="float32")
y_todo = tabla["abandona"].to_numpy(dtype="float32")

X_a, X_b, y_a, y_b = train_test_split(X_todo, y_todo, test_size=0.25,
                                      random_state=SEMILLA, stratify=y_todo)
m_a, d_a = X_a.mean(axis=0), X_a.std(axis=0)
d_a[d_a == 0] = 1.0
X_a, X_b = (X_a - m_a) / d_a, (X_b - m_a) / d_a

keras.utils.set_random_seed(SEMILLA)
trampa = keras.Sequential([
    keras.layers.Input(shape=(10,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
trampa.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
trampa.fit(X_a, y_a, epochs=200, batch_size=32, validation_split=0.2, verbose=0)

auc_trampa = roc_auc_score(y_b, trampa.predict(X_b, verbose=0).ravel())
auc_honesto = [f["AUC prueba"] for f in resultados
               if f["modelo"] == "densa 16-8, 200 epocas"][0]

print(f"Misma red, particion ALEATORIA:  AUC {auc_trampa:.3f}")
print(f"Misma red, particion TEMPORAL:   AUC {auc_honesto:.3f}")
print(f"Diferencia: {auc_trampa - auc_honesto:+.3f}")
print()
print("La sospecha NO se confirma: la particion aleatoria apenas cambia nada.")
print("Asi que el desplazamiento temporal no es lo que hunde a la red, y las")
print("dos causas reales son las dos anteriores: demasiados parametros para")
print("tan pocas filas, y ninguna representacion que aprender.")

> **Una hipótesis razonable que la medición descarta vale exactamente lo mismo que una que
> confirma**, y hay que escribirla igual. Lo que no se puede hacer es mantenerla en el
> informe porque suena bien: sin esta comprobación, "es que la partición es temporal"
> habría quedado como explicación, y sería falsa.

Que la trampa no pague **no la convierte en aceptable**. Sigue siendo entrenar con clientes
de julio para predecir clientes de marzo, que en producción no pasa nunca; si aquí no sube
el AUC es por casualidad de este conjunto, no por un principio. Y es la razón de que la
partición venga ya hecha dentro del CSV: sin ella, el primer
`train_test_split(..., random_state=42)` la convertiría en aleatoria sin que nadie se
diera cuenta.

---

## 5. La evaluación completa, con las herramientas de la UD4

Aunque la red no gane, hay que evaluarla bien: eso es el criterio 2.e, y es lo que pide la
P5.3. Cuatro cosas: la matriz de confusión, la calibración, el desglose por segmento y la
comparación honesta.

In [ ]:
mejor_red = "densa 16-8 regularizada"
p_red = puntuaciones[mejor_red]
umbral_red, _ = mejor_umbral(y_pru, p_red)
pred_red = (p_red >= umbral_red).astype(int)
pred_ref = (recencia_prueba > UMBRAL_REF).astype(int)

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for eje, (titulo, pred) in zip(ejes, [("Punto de referencia UD3", pred_ref),
                                      (mejor_red, pred_red)]):
    mc = confusion_matrix(y_pru, pred)
    imagen = eje.imshow(mc, cmap="Blues")
    for i in range(2):
        for j in range(2):
            eje.text(j, i, mc[i, j], ha="center", va="center",
                     color="white" if mc[i, j] > mc.max() / 2 else "black",
                     fontsize=14)
    eje.set_xticks([0, 1], ["dice: se queda", "dice: se va"])
    eje.set_yticks([0, 1], ["se queda", "se va"])
    eje.set_title(f"{titulo}\nF1 {f1_score(y_pru, pred):.3f}")
fig.suptitle("Las dos matrices, sobre los mismos 100 clientes", y=1.03)
fig.tight_layout()
plt.show()

In [ ]:
# Calibracion: el diagrama de fiabilidad de la UD4, ahora con sklearn.
fig, eje = plt.subplots(figsize=(5.5, 5))
eje.plot([0, 1], [0, 1], "--", color="0.6", lw=1, label="calibracion perfecta")
for nombre in ["logistica (sin capa oculta)", "densa 16-8, 200 epocas"]:
    p = puntuaciones[nombre]
    frecuencia, media_predicha = calibration_curve(y_pru, p, n_bins=5, strategy="quantile")
    eje.plot(media_predicha, frecuencia, "o-", label=nombre)
eje.set_xlabel("probabilidad que dice el modelo")
eje.set_ylabel("frecuencia real de abandono")
eje.set_title("Una puntuacion mal calibrada no se puede leer\ncomo probabilidad, "
              "y el AUC no lo dice")
eje.legend(fontsize=8)
fig.tight_layout()
plt.show()

### El desglose por segmento, y por qué aquí no se puede hacer con el segmento

La UD4 desglosaba por `segmento` —reciente, consolidado, veterano—, que se define por la
antigüedad del cliente. Aquí eso **no funciona**, y el motivo es interesante: la partición
también se hace por antigüedad, así que los cien clientes de prueba son todos `reciente`.
Compruébalo antes de desglosar; es la clase de comprobación que evita una tabla con una
sola fila y tres conclusiones inventadas.

In [ ]:
print(prueba["segmento"].value_counts().to_string())
print()
print("Una sola categoria. El desglose por segmento no dice nada sobre esta")
print("particion, porque la particion ES por antiguedad. Hay que desglosar por")
print("otra variable, y la frecuencia de compra es la que tiene sentido de negocio.")

In [ ]:
# Desglose por frecuencia de compra, en tercios. Es a la vez una comprobacion
# tecnica y una comprobacion de equidad, como en la UD4.
prueba_ = prueba.reset_index(drop=True)
prueba_["grupo"] = pd.qcut(prueba_["frecuencia"], 3,
                           labels=["compra poco", "compra medio", "compra mucho"])

filas = []
for grupo_nombre, grupo in prueba_.groupby("grupo", observed=True):
    idx = grupo.index.to_numpy()
    hay_dos_clases = grupo["abandona"].nunique() == 2
    filas.append({
        "grupo": grupo_nombre,
        "n": len(idx),
        "tasa": grupo["abandona"].mean(),
        "AUC red": roc_auc_score(grupo["abandona"], p_red[idx]) if hay_dos_clases else np.nan,
        "AUC referencia": (roc_auc_score(grupo["abandona"], grupo["recencia_dias"])
                           if hay_dos_clases else np.nan),
    })

desglose = pd.DataFrame(filas)
print(desglose.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print("Un AUC global esconde donde el modelo aporta menos. Con menos de treinta")
print("casos en un grupo la cifra es ruido, y por eso hay que decir siempre la n.")

---

## 6. Qué hay que llevarse, y qué NO

> **El aprendizaje profundo compra representación a cambio de datos. Cuando la
> representación ya está construida y los datos son pocos, el trato es malo.**

Lo que **no** hay que concluir:

- Que las redes neuronales no sirven. Sirven, y en el proyecto de esta unidad ganan por
  goleada. Lo que no son es la opción por defecto.
- Que este conjunto es malo. Es pequeño, y eso es lo normal en un problema de negocio real.
  La mayoría de los problemas de empresa tienen miles de filas, no millones.
- Que haya que dejar de probar. Hay que probar, medir, y **decir el resultado que salga**.

Lo que sí hay que concluir, y es la respuesta al criterio 2.c:

| Pregunta | Respuesta en este caso |
|---|---|
| ¿Hay representación que aprender? | No: la construyó la UD3 a mano |
| ¿Hay datos para aprenderla? | No: 240 filas de entrenamiento |
| ¿Supera el punto de referencia? | No |
| ¿Qué modelo se implementa, entonces? | Una regresión logística, o la regla de la recencia |

**Decirlo con los números delante vale más que un modelo que gana por dos milésimas**, y es
la diferencia entre un informe técnico y un folleto. Es la misma idea del bloque 11 de la
UD4: un límite declarado vale más que un hallazgo extra.

---

## Ejercicios

### Ejercicio 1. El tamaño del conjunto

Repite el experimento del apartado 3 entrenando con el 25 %, el 50 %, el 75 % y el 100 %
de las filas de entrenamiento. Dibuja el AUC de prueba de la logística y el de la densa
16-8 frente al número de filas. ¿Se cruzan las dos curvas? ¿Dónde tendrían que cruzarse?
Esa figura es una **curva de aprendizaje frente al tamaño del conjunto**, y es distinta de
la curva por épocas del cuaderno 05.

### Ejercicio 2. Menos características

Entrena la densa 16-8 usando solo `recencia_dias`, `frecuencia` y `monetario`. ¿Mejora o
empeora respecto de usar las diez? Explica el resultado en términos de capacidad y de
número de parámetros.

### Ejercicio 3. La fuga, fabricada

Añade a `X` una columna calculada con la ventana de resultado —por ejemplo
`pedidos_despues_del_corte`— y vuelve a entrenar. El AUC se dispara. Explica por qué el
resultado es magnífico y por qué es inútil, y relaciónalo con la fila *"resultados
magníficos e increíbles"* de la tabla de diagnóstico del bloque 16.

### Ejercicio 4. El umbral, del coste

El punto de trabajo de este cuaderno sale de maximizar F1, que es una elección cómoda y no
una decisión de negocio. Suponiendo que llamar a un cliente cuesta 3 euros y retener a uno
que se iba vale 40, calcula el umbral que maximiza el beneficio para la red y para el punto
de referencia. ¿Cambia cuál de los dos prefieres? Es el apartado 3 de la P4.2, aplicado
aquí.

### Ejercicio 5. Un modelo que sí gana

Prueba `sklearn.ensemble.GradientBoostingClassifier` con sus opciones por defecto sobre la
misma partición. Los árboles con refuerzo son la respuesta habitual para datos tabulares
pequeños. ¿Gana al punto de referencia? Escribe tres frases sobre por qué un modelo basado
en árboles se comporta distinto de una red densa con estos datos.

### Ejercicio 6. La comparación honesta

Escribe media página dirigida a la dirección de TechStore recomendando qué modelo poner en
producción. Tiene que incluir: la recomendación en la primera línea, una sola cifra de
negocio, la comparación con el punto de referencia, **el segmento donde el modelo aporta
menos**, y dos límites concretos. Sin jerga.

---

## Lo que hay que llevarse de aquí

1. **En TechStore la red no gana al punto de referencia**, y eso es el resultado.
2. **El hueco entre el AUC de entrenamiento y el de prueba es la medida del sobreajuste.**
3. **321 parámetros para 240 filas** es la primera explicación, y se comprueba dividiendo.
4. **Las características ya construidas a mano son la segunda**: no queda representación
   que aprender.
5. **La partición temporal es la tercera**, y hacer la trampa de partir al azar sube el AUC
   de forma medible y falsa.
6. **Cifras medidas sobre poblaciones distintas no se comparan.** El 0,8484 de la UD4 no es
   el AUC de este cuaderno.
7. **Los pesos de una logística con datos normalizados son interpretables**, y aquí dicen
   lo que ya sabíamos: manda la recencia.
8. **Evaluar bien un modelo que pierde es parte del trabajo**, y es el criterio 2.e.